In [1]:
# importing needed packages
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy import stats
# import cartopy.crs as ccrs
# import cartopy.feature as cfeature
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [2]:
# read in the cleaned CSV with data from central valley stations
path = 'E:/Central Valley Fog/CV_rows_codes_incl.csv'
if os.path.exists(path):
    CV_rows = pd.read_csv(path, low_memory = False)
else:
    path = '/Volumes/disk1/Central Valley Fog/CV_rows_codes_incl.csv'
    CV_rows = pd.read_csv(path, low_memory = False)

print(CV_rows)

           STATION_ID  LATITUDE  LONGITUDE                 DATE REPORT_TYPE  \
0         USM00074917   35.1170  -119.3000  1941-12-01 00:00:00         SAO   
1         USM00074917   35.1170  -119.3000  1941-12-01 01:00:00         SAO   
2         USM00074917   35.1170  -119.3000  1941-12-01 02:00:00         SAO   
3         USM00074917   35.1170  -119.3000  1941-12-01 03:00:00         SAO   
4         USM00074917   35.1170  -119.3000  1941-12-01 04:00:00         SAO   
...               ...       ...        ...                  ...         ...   
12877763  USW00024257   40.5175  -122.2986  2026-06-20 04:53:00       FM-15   
12877764  USW00024257   40.5175  -122.2986  2026-06-20 05:53:00       FM-15   
12877765  USW00024257   40.5175  -122.2986  2026-06-20 06:53:00       FM-15   
12877766  USW00024257   40.5175  -122.2986  2026-06-20 07:53:00       FM-15   
12877767  USW00024257   40.5175  -122.2986  2026-06-20 08:53:00       FM-15   

          HourlyPrecipitation HourlyPresentWeatherT

In [3]:
print("CV_rows:", CV_rows.shape) 

CV_rows: (12877768, 15)


In [4]:
CV_rows1 = CV_rows[CV_rows["HourlyPrecipitation"] < 0.03]

print("CV_rows1:", CV_rows1.shape)

CV_rows1: (12476729, 15)


In [5]:
# Unique codes in REPORT_TYPE
report_type_codes = sorted(CV_rows1["REPORT_TYPE"].dropna().astype(str).unique())

# Unique sky-condition codes (e.g., CLR, FEW, SCT, BKN, OVC) parsed from strings like "BKN:06"
hourly_sky_codes = sorted(
    CV_rows1["HourlySkyConditions"]
    .dropna()
    .astype(str)
    .str.findall(r"([A-Z]+)(?=:)")
    .explode()
    .dropna()
    .unique()
)

#find unique dailyweather codes (e.g., CLR, FEW, SCT, BKN, OVC) parsed from strings like "BKN:06"
hourly_weather_codes = sorted(
    CV_rows1["HourlyPresentWeatherType"]
    .dropna()
    .astype(str)
    .str.findall(r"([A-Z]+)(?=:)")
    .explode()
    .dropna()
    .unique()
)

print(f"REPORT_TYPE unique codes ({len(report_type_codes)}):")
print(report_type_codes)

print(f"\nHourlySkyConditions unique codes ({len(hourly_sky_codes)}):")
print(hourly_sky_codes)

print(f"\nHourlyPresentWeatherType unique codes ({len(hourly_weather_codes)}):")
print(hourly_weather_codes)

REPORT_TYPE unique codes (10):
['AUTO', 'FM-12', 'FM-15', 'FM-16', 'SAO', 'SAOSP', 'SMARS', 'SY-MT', 'SYSA', 'WBO_F']

HourlySkyConditions unique codes (5):
['BKN', 'CLR', 'FEW', 'OVC', 'SCT']

HourlyPresentWeatherType unique codes (33):
['BCBR', 'BCFG', 'BLDU', 'BR', 'DS', 'DU', 'DZ', 'FC', 'FG', 'FU', 'FZFG', 'FZRA', 'GR', 'GS', 'HZ', 'IC', 'MIBR', 'MIFG', 'PL', 'PRFG', 'RA', 'SHRA', 'SN', 'SQ', 'SS', 'TSRA', 'UP', 'VCBLDU', 'VCFG', 'VCFU', 'VCRA', 'VCSHRA', 'VCSN']


REPORT_TYPE unique codes + meanings:\
'AUTO' - AUTO\
'FM-12' - FM12-SYNOP-fixed-land-stn\
'FM-15' - FM15-METAR-Aviation-routine-wx\
'FM-16' - FM16-SPECI_Aviation-selected-special-wx\
'SAO' - SAO-Airways-incl-record-specials\
'SAOSP' - SAOSP-Airways-special-excl-record-specials\
'SMARS' - SMARS-Supp-airways-stn\
'SY-MT' - SYMT_Synop-and-METAR-merged\
'SYSA' - SYSA-Synop-and-airways-merged\
'WBO_F' - WBO(?) 

HourlySkyConditions unique codes + meanings:\
'BKN' - broken clouds\
'CLR' - clear \
'FEW' - few clouds \
'OVC' - overcast \
'SCT' - scattered clouds 

HourlyPresentWeatherConditions unique codes + meanings:\
'BCBR' - patches mist\
'BCFG' - patches fog\
'BLDU' - blowing widespread dust\
'BR' - mist\
'DS' - dust storm\
'DU' - widespread dust\
'DZ' drizzle\
'FC' - funnel cloud, waterspout, or tornado\
'FG' - fog\
'FU' - smoke\
'FZFG' - freezing fog\
'FZRA' - freezing rain\
'GR' - hail\
'GS' - small hail and/or snow pellets\
'HZ' - haze\
'IC' - ice crystals\
'MIBR' - shallow mist\
'MIFG' - shallow fog\
'PL' - ice pellets\
'PRFG' - partial fog\
'RA' - rain\
'SHRA' - showers rain\
'SN' - snow\
'SQ' - squalls\
'SS' - sandstorm\
'TSRA' - thunderstorm rain\
'UP'- unknown precipitation \
'VCBLDU' - vicinity blowing widespread dust\
'VCFG' - vicinity fog\
'VCFU' - vicinity smoke\
'VCRA' - vicinity rain\
'VCSHRA' - vicinity showers rain
'VCSN' - vicinity snow

In [6]:
# Fill HourlyVisibility with 0.9 where weather contains "FG" and visibility is NaN
fg_mask = CV_rows1["HourlyPresentWeatherType"].fillna("").astype(str).str.contains("FG", regex=False)
vis_nan_mask = CV_rows1["HourlyVisibility"].isna()

fg_nan_mask = fg_mask & vis_nan_mask

CV_rows1.loc[fg_nan_mask, "HourlyVisibility"] = 0.9

print(f"Rows updated: {fg_nan_mask.sum()}")

Rows updated: 214799


In [7]:
# # save cleaned dataframe
# CV_rows1.to_csv('CV_rows_codes_1.csv', index=False)

In [8]:
foggy_rows = CV_rows1[CV_rows1["HourlyVisibility"] < 1.000]

In [9]:
# Unique codes in REPORT_TYPE
report_type_codes_fog = sorted(CV_rows1["REPORT_TYPE"].dropna().astype(str).unique())

# Unique sky-condition codes (e.g., CLR, FEW, SCT, BKN, OVC) parsed from strings like "BKN:06"
hourly_sky_codes_fog = sorted(
    foggy_rows["HourlySkyConditions"]
    .dropna()
    .astype(str)
    .str.findall(r"([A-Z]+)(?=:)")
    .explode()
    .dropna()
    .unique()
)

#find unique dailyweather codes (e.g., CLR, FEW, SCT, BKN, OVC) parsed from strings like "BKN:06"
hourly_weather_codes_fog = sorted(
    foggy_rows["HourlyPresentWeatherType"]
    .dropna()
    .astype(str)
    .str.findall(r"([A-Z]+)(?=:)")
    .explode()
    .dropna()
    .unique()
)

print(f"REPORT_TYPE unique codes ({len(report_type_codes_fog)}):")
print(report_type_codes_fog)

print(f"\nHourlySkyConditions unique codes ({len(hourly_sky_codes_fog)}):")
print(hourly_sky_codes_fog)

print(f"\nHourlyPresentWeatherType unique codes ({len(hourly_weather_codes_fog)}):")
print(hourly_weather_codes_fog)

REPORT_TYPE unique codes (10):
['AUTO', 'FM-12', 'FM-15', 'FM-16', 'SAO', 'SAOSP', 'SMARS', 'SY-MT', 'SYSA', 'WBO_F']

HourlySkyConditions unique codes (5):
['BKN', 'CLR', 'FEW', 'OVC', 'SCT']

HourlyPresentWeatherType unique codes (21):
['BCBR', 'BCFG', 'BLDU', 'BR', 'DS', 'DZ', 'FG', 'FU', 'FZFG', 'FZRA', 'HZ', 'MIBR', 'MIFG', 'PL', 'PRFG', 'RA', 'SN', 'SQ', 'SS', 'UP', 'VCFG']


In [10]:
fg_in_foggy = foggy_rows["HourlyPresentWeatherType"].fillna("").astype(str).str.contains("FG", regex=False)

print("foggy_rows with 'FG' in HourlyPresentWeatherType:", fg_in_foggy.sum())
print("foggy_rows without 'FG' in HourlyPresentWeatherType:", (~fg_in_foggy).sum())

foggy_rows with 'FG' in HourlyPresentWeatherType: 461073
foggy_rows without 'FG' in HourlyPresentWeatherType: 21565


In [11]:
foggy_rows.head(50)

,STATION_ID,LATITUDE,LONGITUDE,DATE,REPORT_TYPE,HourlyPrecipitation,HourlyPresentWeatherType,HourlySkyConditions,HourlyVisibility,year,month,day,hour,minute,STATION
6,USM00074917,35.117,-119.3,1941-12-01 06:00:00,SAO,0.0,||FG,NaN,0.2,1941,12,1,6,0,GARDNER AAF
7,USM00074917,35.117,-119.3,1941-12-01 07:00:00,SAO,0.0,||FG,NaN,0.0,1941,12,1,7,0,GARDNER AAF
8,USM00074917,35.117,-119.3,1941-12-01 08:00:00,SAO,0.0,||FG,NaN,0.0,1941,12,1,8,0,GARDNER AAF
9,USM00074917,35.117,-119.3,1941-12-01 09:00:00,SAO,0.0,||FG,NaN,0.2,1941,12,1,9,0,GARDNER AAF
10,USM00074917,35.117,-119.3,1941-12-01 10:00:00,SAO,0.0,||FG,OVC:08,0.8,1941,12,1,10,0,GARDNER AAF
26,USM00074917,35.117,-119.3,1941-12-02 02:00:00,SAO,0.0,||FG,OVC:08,0.8,1941,12,2,2,0,GARDNER AAF
27,USM00074917,35.117,-119.3,1941-12-02 03:00:00,SAO,0.0,||FG,NaN,0.0,1941,12,2,3,0,GARDNER AAF
28,USM00074917,35.117,-119.3,1941-12-02 04:00:00,SAO,0.0,||FG,NaN,0.0,1941,12,2,4,0,GARDNER AAF
29,USM00074917,35.117,-119.3,1941-12-02 05:00:00,SAO,0.0,||FG,FEW:02,0.8,1941,12,2,5,0,GARDNER AAF
31,USM00074917,35.117,-119.3,1941-12-02 07:00:00,SAO,0.0,||FG,SCT:03,0.4,1941,12,2,7,0,GARDNER AAF


In [12]:
# Rows in foggy_rows where HourlyPresentWeatherType contains no letters
no_letter_mask = ~CV_rows1['HourlyPresentWeatherType'].fillna("").astype(str).str.contains(r"[A-Za-z]", regex=True)
no_letter_rows = CV_rows1[no_letter_mask].copy()

print("Rows with no letters in HourlyPresentWeatherType:", no_letter_rows.shape[0])
display(no_letter_rows)

Rows with no letters in HourlyPresentWeatherType: 10109904


,STATION_ID,LATITUDE,LONGITUDE,DATE,REPORT_TYPE,HourlyPrecipitation,HourlyPresentWeatherType,HourlySkyConditions,HourlyVisibility,year,month,day,hour,minute,STATION
0,USM00074917,35.1170,-119.3000,1941-12-01 00:00:00,SAO,0.0,||00,BKN:06,16.093,1941,12,1,0,0,GARDNER AAF
1,USM00074917,35.1170,-119.3000,1941-12-01 01:00:00,SAO,0.0,||00,OVC:08,16.093,1941,12,1,1,0,GARDNER AAF
2,USM00074917,35.1170,-119.3000,1941-12-01 02:00:00,SAO,0.0,||00,OVC:08,16.093,1941,12,1,2,0,GARDNER AAF
3,USM00074917,35.1170,-119.3000,1941-12-01 03:00:00,SAO,0.0,||00,OVC:08,16.093,1941,12,1,3,0,GARDNER AAF
4,USM00074917,35.1170,-119.3000,1941-12-01 04:00:00,SAO,0.0,||00,OVC:08,16.093,1941,12,1,4,0,GARDNER AAF
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12877763,USW00024257,40.5175,-122.2986,2026-06-20 04:53:00,FM-15,0.0,NaN,CLR:00,16.090,2026,6,20,4,53,REDDING AP
12877764,USW00024257,40.5175,-122.2986,2026-06-20 05:53:00,FM-15,0.0,NaN,CLR:00,16.090,2026,6,20,5,53,REDDING AP
12877765,USW00024257,40.5175,-122.2986,2026-06-20 06:53:00,FM-15,0.0,NaN,CLR:00,16.090,2026,6,20,6,53,REDDING AP
12877766,USW00024257,40.5175,-122.2986,2026-06-20 07:53:00,FM-15,0.0,NaN,CLR:00,16.090,2026,6,20,7,53,REDDING AP


In [13]:
no_letter_rows = no_letter_rows[no_letter_rows["HourlyVisibility"] < 1].copy()

print("Rows with no letters in HourlyPresentWeatherType and visibility < 1:", no_letter_rows.shape[0])
no_letter_rows.head(-15)

Rows with no letters in HourlyPresentWeatherType and visibility < 1: 16025


,STATION_ID,LATITUDE,LONGITUDE,DATE,REPORT_TYPE,HourlyPrecipitation,HourlyPresentWeatherType,HourlySkyConditions,HourlyVisibility,year,month,day,hour,minute,STATION
1441,USM00074917,35.1170,-119.3000,1942-01-30 01:00:00,SAO,0.0,||00,OVC:08,0.000,1942,1,30,1,0,GARDNER AAF
1442,USM00074917,35.1170,-119.3000,1942-01-30 02:00:00,SAO,0.0,||00,OVC:08,0.000,1942,1,30,2,0,GARDNER AAF
1443,USM00074917,35.1170,-119.3000,1942-01-30 03:00:00,SAO,0.0,||00,BKN:05 FEW:02,0.000,1942,1,30,3,0,GARDNER AAF
1444,USM00074917,35.1170,-119.3000,1942-01-30 04:00:00,SAO,0.0,||00,FEW:02 18.00 SCT:03,0.000,1942,1,30,4,0,GARDNER AAF
1445,USM00074917,35.1170,-119.3000,1942-01-30 05:00:00,SAO,0.0,||00,OVC:08,0.000,1942,1,30,5,0,GARDNER AAF
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12706054,USW00024257,40.5175,-122.2986,2010-12-12 03:02:00,FM-16,0.0,NaN,FEW:02 0.30 BKN:07 2.44,0.805,2010,12,12,3,2,REDDING AP
12706055,USW00024257,40.5175,-122.2986,2010-12-12 03:10:00,FM-16,0.0,NaN,FEW:02 0.30 BKN:07 2.44,0.402,2010,12,12,3,10,REDDING AP
12706056,USW00024257,40.5175,-122.2986,2010-12-12 03:22:00,FM-16,0.0,NaN,BKN:07 0.30 OVC:08 2.44,0.805,2010,12,12,3,22,REDDING AP
12706057,USW00024257,40.5175,-122.2986,2010-12-12 03:32:00,FM-16,0.0,NaN,BKN:07 0.30 OVC:08 2.44,0.402,2010,12,12,3,32,REDDING AP


In [14]:
# Keep rows with FG or no letters in HourlyPresentWeatherType from CV_rows1
fg_mask = CV_rows1['HourlyPresentWeatherType'].fillna("").astype(str).str.contains("FG", regex=False)
combined_mask = fg_mask | no_letter_mask

CV_rows1_filtered = CV_rows1[combined_mask].copy()
print(f"CV_rows1_filtered shape: {CV_rows1_filtered.shape}")

CV_rows1_filtered shape: (10961946, 15)


In [15]:
# CV_rows1_filtered.to_csv('CV_rows_codes_filtered.csv', index=False)

In [16]:
zeros_in_foggy = foggy_rows["HourlyPresentWeatherType"].fillna("").astype(str).str.contains("0", regex=False)

print("zero_rows with '0' in HourlyPresentWeatherType:", zeros_in_foggy.sum())
print("zero_rows without '0' in HourlyPresentWeatherType:", (~zeros_in_foggy).sum())

zero_rows with '0' in HourlyPresentWeatherType: 21993
zero_rows without '0' in HourlyPresentWeatherType: 460645


In [17]:
# Keep foggy rows that contain either FG or 00 in the weather code
weather_str = foggy_rows["HourlyPresentWeatherType"].fillna("").astype(str)
keep_mask = weather_str.str.contains("FG", regex=False) | weather_str.str.contains("00", regex=False)

removed_count = (~keep_mask).sum()
foggy_rows = foggy_rows[keep_mask].copy()

print(f"Removed rows without FG/00: {removed_count}")
print("foggy_rows (FG or 00):", foggy_rows.shape)

Removed rows without FG/00: 20641
foggy_rows (FG or 00): (461997, 15)


In [18]:
non_fg_foggy = foggy_rows[~fg_in_foggy]
print(non_fg_foggy)

           STATION_ID  LATITUDE  LONGITUDE                 DATE REPORT_TYPE  \
1441      USM00074917   35.1170  -119.3000  1942-01-30 01:00:00         SAO   
1442      USM00074917   35.1170  -119.3000  1942-01-30 02:00:00         SAO   
1443      USM00074917   35.1170  -119.3000  1942-01-30 03:00:00         SAO   
1444      USM00074917   35.1170  -119.3000  1942-01-30 04:00:00         SAO   
1445      USM00074917   35.1170  -119.3000  1942-01-30 05:00:00         SAO   
...               ...       ...        ...                  ...         ...   
12610037  USW00024257   40.5175  -122.2986  2002-08-19 11:53:00       FM-15   
12610038  USW00024257   40.5175  -122.2986  2002-08-19 12:53:00       FM-15   
12618310  USW00024257   40.5175  -122.2986  2003-05-15 12:51:00       FM-15   
12625505  USW00024257   40.5175  -122.2986  2004-01-04 12:53:00       FM-15   
12642599  USW00024257   40.5175  -122.2986  2005-06-15 20:53:00       FM-15   

          HourlyPrecipitation HourlyPresentWeatherT

C:\Users\Work Account\AppData\Local\Temp\ipykernel_15544\19031442.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  non_fg_foggy = foggy_rows[~fg_in_foggy]


In [19]:
non_fg_weather_codes = sorted(
    non_fg_foggy["HourlyPresentWeatherType"]
    .dropna()
    .astype(str)
    .str.findall(r"([A-Z]+)(?=:)")
    .explode()
    .dropna()
    .unique()
)

print(f"Unique HourlyPresentWeatherType codes in non_fg_foggy ({len(non_fg_weather_codes)}):")
print(non_fg_weather_codes)

Unique HourlyPresentWeatherType codes in non_fg_foggy (2):
['BR', 'HZ']


'BLDU' - blowing dust\
'BR' - mist\
'DS' - dust storm\
'DZ' - drizzle\
'FU' - smoke\
'HZ' - haze\
'MIBR' - shallow mist\
'PL' -  ice pellets\
'RA' - rain\
'SN' - snow\
'SS' - sandstorm\
'UP' - unknown precipitation

In [20]:
# Keep only rows with FG code in foggy_rows
foggy_rows = foggy_rows[fg_in_foggy].copy()

print(f"Removed rows without FG: {(~fg_in_foggy).sum()}")
print("foggy_rows (FG only):", foggy_rows.shape)

Removed rows without FG: 21565
foggy_rows (FG only): (461073, 15)


C:\Users\Work Account\AppData\Local\Temp\ipykernel_15544\1842262607.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  foggy_rows = foggy_rows[fg_in_foggy].copy()


In [21]:
non_fg_foggy.head(50)

,STATION_ID,LATITUDE,LONGITUDE,DATE,REPORT_TYPE,HourlyPrecipitation,HourlyPresentWeatherType,HourlySkyConditions,HourlyVisibility,year,month,day,hour,minute,STATION
1441,USM00074917,35.11700,-119.3,1942-01-30 01:00:00,SAO,0.0,||00,OVC:08,0.0,1942,1,30,1,0,GARDNER AAF
1442,USM00074917,35.11700,-119.3,1942-01-30 02:00:00,SAO,0.0,||00,OVC:08,0.0,1942,1,30,2,0,GARDNER AAF
1443,USM00074917,35.11700,-119.3,1942-01-30 03:00:00,SAO,0.0,||00,BKN:05 FEW:02,0.0,1942,1,30,3,0,GARDNER AAF
1444,USM00074917,35.11700,-119.3,1942-01-30 04:00:00,SAO,0.0,||00,FEW:02 18.00 SCT:03,0.0,1942,1,30,4,0,GARDNER AAF
1445,USM00074917,35.11700,-119.3,1942-01-30 05:00:00,SAO,0.0,||00,OVC:08,0.0,1942,1,30,5,0,GARDNER AAF
1446,USM00074917,35.11700,-119.3,1942-01-30 06:00:00,SAO,0.0,||00,BKN:06,0.0,1942,1,30,6,0,GARDNER AAF
1447,USM00074917,35.11700,-119.3,1942-01-30 07:00:00,SAO,0.0,||00,OVC:08,0.0,1942,1,30,7,0,GARDNER AAF
1448,USM00074917,35.11700,-119.3,1942-01-30 08:00:00,SAO,0.0,||00,BKN:06 24.00,0.0,1942,1,30,8,0,GARDNER AAF
1449,USM00074917,35.11700,-119.3,1942-01-30 09:00:00,SAO,0.0,||00,BKN:05 SCT:03,0.0,1942,1,30,9,0,GARDNER AAF
1450,USM00074917,35.11700,-119.3,1942-01-30 10:00:00,SAO,0.0,||00,FEW:02 15.00 FEW:02,0.0,1942,1,30,10,0,GARDNER AAF


In [22]:
# foggy_rows.to_csv('E:/Central Valley Fog/FG_rows.csv', index=False)
# print("Saved foggy rows to CSV")

In [23]:
print((~fg_in_foggy).sum())

21565
